# Pilot Analysis — LoRA Gemma-2-9B parsing self-repair SBN

**Model:** `Shrikes/sbn-gemma2-9b-lora-pmb` (LoRA on PMB 5.1.0 gold).  
**Data:** `Shrikes/self_repair_parsing_pilot_data` (836 gold sentences × 7 conditions).  
**Scoring:** smatch++ (flipz357/smatchpp), via `evaluate_smatchpp.py`.

Each of the 836 source sentences is parsed under 7 conditions and scored against the **same** gold SBN (`mr`):

| group | conditions |
|---|---|
| baseline | `gold` |
| repair (plain) | `repair_head`, `repair_mid`, `repair_tail` |
| repair (+interregnum "I mean") | `repair_head_interrug`, `repair_mid_interrug`, `repair_tail_interrug` |

`gold` measures base parser quality; the repair conditions measure degradation when an un-trained-for self-repair is forced through the parser.

### Status codes
`success` = both gold & pred SBN → Penman → smatch++ OK. `ill_formed` = pred rejected by strict mode (impossible indices). `parse_error` = pred not parseable. `gold_error` = gold itself failed (should be ~0). `smatch_error` = scoring crashed.

### Two F1 metrics
* **F1 (A, success-only):** mean over rows where `status == success`. Hides failure cost.
* **F1 (B, penalized):** failures counted as F1 = 0, mean over **all** rows. This is the headline degradation metric — a parser that emits ill-formed graphs on repair input is penalized.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)

# Path to the scored LONG table produced by evaluate_smatchpp.py
SCORED_PATH = Path("preds_long_scored.parquet")
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

CONDITION_ORDER = [
    "gold",
    "repair_head", "repair_mid", "repair_tail",
    "repair_head_interrug", "repair_mid_interrug", "repair_tail_interrug",
]
POSITION_ORDER = ["head", "mid", "tail"]
STATUS_ORDER = ["success", "ill_formed", "parse_error", "gold_error", "smatch_error"]

In [ ]:
df = pd.read_parquet(SCORED_PATH) if SCORED_PATH.suffix == ".parquet" else pd.read_csv(SCORED_PATH, sep="\t")
df["f1"] = pd.to_numeric(df["f1"], errors="coerce")
df["condition"] = pd.Categorical(df["condition"], categories=CONDITION_ORDER, ordered=True)

# Derived helpers for grouped views.
def position(c):
    for p in POSITION_ORDER:
        if f"_{p}" in c: return p
    return "—"
df["position"] = df["condition"].astype(str).map(position)
df["has_interrug"] = df["condition"].astype(str).str.contains("interrug")
# Penalized F1 (B): non-success -> 0.0
df["f1_pen"] = df["f1"].where(df["status"] == "success", 0.0)

print(f"rows: {len(df)}  | conditions: {df['condition'].nunique()}  | ids: {df['id'].nunique()}")
print("status breakdown:")
print(df["status"].value_counts().to_string())

## 1. Summary table (per condition)

In [ ]:
def summarize(g):
    n = len(g)
    ok = (g["status"] == "success").sum()
    succ_f1 = g.loc[g["status"] == "success", "f1"]
    row = {
        "N": n,
        "success_n": ok,
        "success_%": round(100 * ok / n, 1) if n else 0,
        "F1_A_succ_mean": round(succ_f1.mean(), 4) if len(succ_f1) else np.nan,
        "F1_A_succ_median": round(succ_f1.median(), 4) if len(succ_f1) else np.nan,
        "F1_B_penalized_mean": round(g["f1_pen"].mean(), 4),
    }
    for s in STATUS_ORDER:
        row[s] = int((g["status"] == s).sum())
    return pd.Series(row)

summary = df.groupby("condition", observed=True).apply(summarize).reindex(CONDITION_ORDER)
summary.to_csv(FIG_DIR / "summary_table.csv")
summary

## 2. Success rate by condition

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#555555"] + sns.color_palette("muted", 6)
ax.bar(summary.index, summary["success_%"], color=colors)
ax.axhline(summary.loc["gold", "success_%"], ls="--", c="#555555", lw=1, label="gold baseline")
ax.set_ylabel("success %"); ax.set_title("Parse success rate by condition")
ax.set_xticklabels(summary.index, rotation=30, ha="right"); ax.legend()
for i, v in enumerate(summary["success_%"]):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_success_rate.png", dpi=150); plt.show()

## 3. F1 distribution (success-only) by condition

In [ ]:
succ = df[df["status"] == "success"]
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=succ, x="condition", y="f1", order=CONDITION_ORDER, cut=0, inner="quartile", ax=ax)
ax.set_title("F1 distribution among successfully-parsed rows (metric A)")
ax.set_xticklabels(CONDITION_ORDER, rotation=30, ha="right"); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_f1_violin.png", dpi=150); plt.show()

## 4. Penalized F1 (B) — headline degradation metric
Failures count as 0. This is the number to report for "how much does forced self-repair parsing hurt".

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(summary.index, summary["F1_B_penalized_mean"], color=colors)
ax.axhline(summary.loc["gold", "F1_B_penalized_mean"], ls="--", c="#555555", lw=1, label="gold baseline")
ax.set_ylabel("mean F1 (penalized)"); ax.set_ylim(0, 1.02)
ax.set_title("Penalized mean F1 by condition (failures = 0)")
ax.set_xticklabels(summary.index, rotation=30, ha="right"); ax.legend()
for i, v in enumerate(summary["F1_B_penalized_mean"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_f1_penalized.png", dpi=150); plt.show()

## 5. Failure breakdown (stacked) by condition

In [ ]:
fail_cols = [s for s in STATUS_ORDER if s != "success"]
fails = summary[fail_cols]
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(summary))
pal = sns.color_palette("Reds", len(fail_cols))
for col, c in zip(fail_cols, pal):
    ax.bar(summary.index, fails[col], bottom=bottom, label=col, color=c)
    bottom += fails[col].values
ax.set_ylabel("# rows"); ax.set_title("Failure-type breakdown by condition")
ax.set_xticklabels(summary.index, rotation=30, ha="right"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_failure_breakdown.png", dpi=150); plt.show()

## 6. Position × interregnum interaction
Does reparandum position (head/mid/tail) and the presence of "I mean" interact? Uses penalized F1.

In [ ]:
rep = df[df["position"].isin(POSITION_ORDER)].copy()
piv = rep.groupby(["position", "has_interrug"], observed=True)["f1_pen"].mean().unstack()
piv = piv.reindex(POSITION_ORDER)
piv.columns = ["plain" if c is False else "+interregnum" for c in piv.columns]
ax = piv.plot(kind="bar", figsize=(8, 5), rot=0)
ax.axhline(summary.loc["gold", "F1_B_penalized_mean"], ls="--", c="#555555", lw=1, label="gold baseline")
ax.set_ylabel("mean F1 (penalized)"); ax.set_xlabel("reparandum position")
ax.set_title("Penalized F1 by position × interregnum"); ax.set_ylim(0, 1.02); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_position_interreg.png", dpi=150); plt.show()
piv

## 7. Per-item paired view: gold vs each repair condition
How far does each repair condition fall below the same item's gold score? (Joined on `id`, penalized F1.)

In [ ]:
wide = df.pivot_table(index="id", columns="condition", values="f1_pen", observed=True)
deltas = wide.drop(columns=["gold"]).sub(wide["gold"], axis=0)
delta_summary = deltas.mean().reindex([c for c in CONDITION_ORDER if c != "gold"]).rename("mean ΔF1 vs gold")
print("Mean per-item drop from gold (penalized F1):")
print(delta_summary.to_string())
fig, ax = plt.subplots(figsize=(9, 5))
delta_summary.plot(kind="barh", ax=ax, color=sns.color_palette("muted", 6))
ax.axvline(0, c="#333", lw=1); ax.set_xlabel("mean ΔF1 (condition − gold)")
ax.set_title("Per-item degradation from gold")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig_delta_vs_gold.png", dpi=150); plt.show()

## Notes for the thesis write-up
* Report **both** F1 metrics; the gap between A and B quantifies how much of the damage is *ill-formed output* vs *wrong-but-valid* structure.
* `gold` success% < 100 is the base parser's own error floor; degradation should be read relative to it, not to 1.0.
* Interregnum ("I mean") effect is the difference between the two bars at each position in §6.
* Optional: smatch++ has built-in bootstrap CIs if condition differences need significance testing.